# 07 - Full Pipeline: End-to-End Lesson Generation
**Module:** `src/core/generator.py`

## What this notebook demonstrates
The complete Bantrly lesson generation pipeline -- from a teacher's
three inputs to a validated, saved, research-grounded lesson JSON.

## The full sequence
```
Teacher inputs: Grade Band + ELA Domain + Theme
        ↓
1. Pre-generation guardrail checks
        ↓
2. Prompt construction (system + few-shot + user)
        ↓
3. Groq API call (llama-3.3-70b-versatile)
        ↓
4. Output validation (strip fences → parse → check fields → guardrails)
        ↓
5. Save to data/generated/
        ↓
6. Register in deduplication registry
        ↓
Complete lesson JSON
```

## What we'll do
1. Generate a lesson and walk through every step
2. Inspect the generated lesson in full
3. Generate lessons across multiple grade bands
4. Demonstrate the deduplication registry growing
5. Show what happens when bad inputs are provided

In [1]:
import sys
sys.path.insert(0, '..')

from src.core.generator import LessonGenerator
from src.utils.file_handler import load_lesson, load_registry
from src.guardrails.checks import run_pre_checks
from src.core.skill_selector import get_next_skill, get_coverage_report

import json


## Part 1: Initialising the Generator
The generator loads your Groq API key from `.env` and
initialises the Groq client. If your key is missing or
invalid, it fails here -- not mid-generation.

In [2]:
gen = LessonGenerator(verbose=True)

[generator] LessonGenerator initialised ✅
[generator] Model: llama-3.1-8b-instant


## Part 2: Previewing the Prompt
Before making any API call, let's see exactly what
the model will receive for our first generation request, including the skill that has been auto-selected from
the taxonomy.
No tokens spent -- just inspection.

In [3]:
# Preview the prompt without making an API call
gen.preview_prompt(
    grade_band = "3-5",
    ela_domain = "Speaking",
    theme      = "Ocean Ecosystems"
)

[preview] Auto-selected skill: Respond to a peer's idea by agreeing or disagreeing with a reason

Total messages: 4

[1] SYSTEM
----------------------------------------
You are an expert K-12 ELA lesson designer for Bantrly, a voice-based
language learning platform for students. Your job is to generate a single,
complete, structured lesson in JSON format.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LESSON DESIGN PRINCIPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Ev...

[2] USER
----------------------------------------
Here is a complete example of a well-designed Bantrly lesson for grade band 3-5. Study its structure, depth, and how it follows all the design principles:

{
  "lesson_id": "L-35-SPK-005",
  "metadata": {
    "grade_band": "3-5",
    "ela_domain": "Speaking",
    "lesson_type": "Mission Brief",
    ...

[3] ASSISTANT
----------------------------------------
Understood. I have studied the example lesson carefully. I will follow the same structure, depth, narrative quality,

In [4]:
# See which skill will be targeted before we generate
next_skill = get_next_skill("3-5", "Speaking")
print(f"Next skill to cover: {next_skill}")

report = get_coverage_report("3-5", "Speaking")
print(f"Coverage before generation: {report['covered_count']}/{report['total']} skills covered")
print(f"Remaining: {report['remaining']}")

Next skill to cover: Respond to a peer's idea by agreeing or disagreeing with a reason
Coverage before generation: 5/5 skills covered
Remaining: []


## Part 3: Generating Our First Lesson
Now we make the real API call.
Printing the logs as it completes.

In [5]:
lesson = gen.generate(
    grade_band = "3-5",
    ela_domain = "Speaking",
    theme      = "Ocean Ecosystems"
)


[generator] Starting lesson generation
[generator] Grade: 3-5 | Domain: Speaking | Theme: Ocean Ecosystems

[generator] Step 1 — Running pre-generation checks...
[checks] All pre-generation checks passed ✅
[generator] Step 2 — Checking deduplication...

[generator] Step 3 — Selecting skill...
[generator] Auto-selected skill: Respond to a peer's idea by agreeing or disagreeing with a reason

[generator] Step 4 — Building prompt...

[generator] Step 5 — Calling Groq API (llama-3.1-8b-instant)...
[generator] Attempt 1/3...
[generator] Received response — 4,098 chars

[generator] Step 6 — Validating output...
[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] ⚠️  practice was a dict with 'prompts' key — unwrapping...
[validator] Auto-corrected practice → 2 prompts
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Respond to a peer's i

In [6]:
# Confirm the skill in the lesson matches what the selector picked
print(f"Skill selected  : {next_skill}")
print(f"Skill in lesson : {lesson['metadata']['primary_skill']}")
print()

match = next_skill == lesson['metadata']['primary_skill']
print(f"Exact match: {'✅ Yes' if match else '⚠️  No — LLM modified the skill string'}")

Skill selected  : Respond to a peer's idea by agreeing or disagreeing with a reason
Skill in lesson : Respond to a peer's idea by agreeing or disagreeing with a reason

Exact match: ✅ Yes


In [7]:
# High-level summary
print("=== LESSON SUMMARY ===\n")
print(f"ID             : {lesson['lesson_id']}")
print(f"Grade Band     : {lesson['metadata']['grade_band']}")
print(f"ELA Domain     : {lesson['metadata']['ela_domain']}")
print(f"Lesson Type    : {lesson['metadata']['lesson_type']}")
print(f"Theme          : {lesson['metadata']['theme']}")
print(f"Primary Skill  : {lesson['metadata']['primary_skill']}")
print(f"Voice Markers  : {lesson['metadata']['voice_markers']}")
print(f"Duration       : {lesson['metadata']['estimated_duration_minutes']} minutes")
print(f"CCSS Anchor    : {lesson['metadata']['ccss_anchor']}")

=== LESSON SUMMARY ===

ID             : L-35-SPK-022
Grade Band     : 3-5
ELA Domain     : Speaking
Lesson Type    : Mission Brief
Theme          : Ocean Ecosystems
Primary Skill  : Respond to a peer's idea by agreeing or disagreeing with a reason
Voice Markers  : ['Fluency & Fillers', 'Speaking Rate']
Duration       : 8 minutes
CCSS Anchor    : CCSS.ELA-Literacy.SL.4.1 — Engage effectively in a range of discussions (one-on-one, in groups, written) to support a topic, text, or issue, by conveying well-defined and thoughtful ideas and concepts as well as presenting clear and respectful responses.


## Part 4: Inspecting the Generated Lesson
Let's read the lesson carefully -- not just the metadata
but the actual content. This is the quality check.

In [8]:
# High-level summary
print("=== LESSON SUMMARY ===\n")
print(f"ID             : {lesson['lesson_id']}")
print(f"Grade Band     : {lesson['metadata']['grade_band']}")
print(f"ELA Domain     : {lesson['metadata']['ela_domain']}")
print(f"Lesson Type    : {lesson['metadata']['lesson_type']}")
print(f"Theme          : {lesson['metadata']['theme']}")
print(f"Primary Skill  : {lesson['metadata']['primary_skill']}")
print(f"Voice Markers  : {lesson['metadata']['voice_markers']}")
print(f"Duration       : {lesson['metadata']['estimated_duration_minutes']} minutes")
print(f"CCSS Anchor    : {lesson['metadata']['ccss_anchor']}")

=== LESSON SUMMARY ===

ID             : L-35-SPK-022
Grade Band     : 3-5
ELA Domain     : Speaking
Lesson Type    : Mission Brief
Theme          : Ocean Ecosystems
Primary Skill  : Respond to a peer's idea by agreeing or disagreeing with a reason
Voice Markers  : ['Fluency & Fillers', 'Speaking Rate']
Duration       : 8 minutes
CCSS Anchor    : CCSS.ELA-Literacy.SL.4.1 — Engage effectively in a range of discussions (one-on-one, in groups, written) to support a topic, text, or issue, by conveying well-defined and thoughtful ideas and concepts as well as presenting clear and respectful responses.


In [9]:
# Read the full lesson flow
flow = lesson["lesson_flow"]

print("=== HOOK ===")
print(flow["hook"]["content"])

print("\n=== MODEL STAGE ===")
print(flow["model"]["skill_named_explicitly"])
print()
print(flow["model"]["content"])

print("\n=== PRACTICE PROMPTS ===")
for p in flow["practice"]:
    print(f"\n[{p['prompt_id']}] Type: {p['type']}")
    print(f"Prompt  : {p['text']}")
    if p.get("scaffold"):
        print(f"Scaffold: {p['scaffold']}")

print("\n=== REFLECT ===")
print(f"Voice marker focus : {flow['reflect']['voice_marker_focus']}")
print(f"Positive signal    : {flow['reflect']['positive_signal']}")
print(f"Growth signal      : {flow['reflect']['growth_signal']}")

=== HOOK ===
INCOMING TRANSMISSION. Agent, are you there? This is Mission Control. We've received a report about a unique ocean ecosystem. One of our team members, Sarah, thinks sea turtles help protect coral reefs by eating seaweed. Another team member, Alex, disagrees. We need you to review the data and decide: do you agree or disagree with Sarah, and why?

=== MODEL STAGE ===
Today we are practicing: responding to a peer's idea by agreeing or disagreeing with a reason.

Here is what a clear response to a peer's idea sounds like: 'I agree with Sarah that sea turtles help protect coral reefs. That's because turtles eat seaweed that can grow too thick and choke the reef. If they didn't eat it, the coral might not be able to get the sunlight it needs to survive.' Notice: each reason connects to the peer's idea, and details explain *why*, not just *what*.

=== PRACTICE PROMPTS ===

[P1] Type: supported
Prompt  : Your turn, Agent. Review the data and decide: do you agree or disagree with 

KeyError: 'voice_marker_focus'

In [ ]:
# Check the guardrail flags
print("=== GUARDRAIL FLAGS ===\n")
for check, result in lesson["guardrail_flags"].items():
    icon = "✅" if result["status"] == "pass" else "⚠️"
    print(f"{icon} {check}")
    print(f"   {result['message']}")
    print()

=== GUARDRAIL FLAGS ===

✅ single_skill_check
   Single skill confirmed: 'Explain a process step by step in your own words'

✅ vocabulary_ceiling_check
   All prompts within 3-5 vocabulary ceiling (60 words).

✅ cognitive_load_check
   Cognitive load check passed for grade band '3-5'.

✅ cultural_bias_check
   No cultural bias terms detected.



In [ ]:
# Print the full lesson JSON
print("=== FULL LESSON JSON ===\n")
print(json.dumps(lesson, indent=2))

=== FULL LESSON JSON ===

{
  "lesson_id": "L-35-SPK-012",
  "metadata": {
    "grade_band": "3-5",
    "ela_domain": "Speaking",
    "lesson_type": "Mission Brief",
    "theme": "Ocean Ecosystems",
    "primary_skill": "Explain a process step by step in your own words",
    "voice_markers": [
      "Fluency & Fillers",
      "Speaking Rate"
    ],
    "estimated_duration_minutes": 6,
    "ccss_anchor": "CCSS.ELA-Literacy.SL.4.4 \u2014 Report on a topic or text, tell a story, or recount an experience in an organized manner, using appropriate facts and relevant, descriptive details, speaking clearly at an understandable pace.",
    "design_notes": "Mission Brief format embeds speaking task within a narrative role \u2014 student becomes the expert, lowering the affective barrier to speaking. Ocean ecosystems theme is age-appropriate and culturally inclusive. Two sentence starters for P1 only; P2 is independent to push toward transfer."
  },
  "lesson_flow": {
    "hook": {
      "duratio

## Part 5: Generating Across Grade Bands
Let's generate one lesson per grade band to demonstrate
the system producing differentiated content automatically.

This is the range the brief asked for -- same pipeline,
fundamentally different outputs based on grade band rules.

In [ ]:
# Generate a lesson for each grade band
# Using verbose=False to keep output clean

gen_quiet = LessonGenerator(verbose=False)

grade_requests = [
    ("K-2",  "Speaking",           "Nature & Animals"),
    ("6-8",  "Speaking",           "Climate Change"),
    ("9-12", "Reading → Speaking", "Artificial Intelligence Ethics"),
]

generated_lessons = []

for grade, domain, theme in grade_requests:
    skill = get_next_skill(grade, domain)
    print(f"Generating : {grade} | {domain} | {theme}")
    print(f"Skill      : {skill}")
    l = gen_quiet.generate(grade_band=grade, ela_domain=domain, theme=theme)
    generated_lessons.append(l)
    print(f"✅ {l['lesson_id']} — {l['metadata']['primary_skill'][:60]}")
    print()

Generating : K-2 | Speaking | Nature & Animals
Skill      : Describe a person, place, or thing using sensory details
[checks] All pre-generation checks passed ✅
[validator] Starting validation pipeline...
[validator] Step 1 — Code fences stripped ✅
[validator] Step 2 — JSON parsed successfully ✅
[validator] Step 3 — Required fields present ✅
[checks] single_skill_check: ✅ [PASS] Single skill confirmed: 'Describe a person, place, or thing using sensory details'
[checks] vocabulary_ceiling_check: ✅ [PASS] All prompts within K-2 vocabulary ceiling (30 words).
[checks] cognitive_load_check: ✅ [PASS] Cognitive load check passed for grade band 'K-2'.
[checks] cultural_bias_check: ✅ [PASS] No cultural bias terms detected.
[validator] Step 4 — Guardrail checks complete ✅
[validator] Validation passed for lesson: L-K2-SPK-002
[file_handler] Saved lesson → D:\projects\bantrly-lesson-generator\data\generated\L-K2-SPK-004.json
[file_handler] Registered combo → K-2 | Nature & Animals | Describe a p

In [ ]:
# Compare the generated lessons side by side
print("=== CROSS GRADE BAND COMPARISON ===\n")
print(f"{'Grade':<8} {'Type':<15} {'Skill':<55} {'Scaffolded?'}")
print(f"{'-'*8:<8} {'-'*15:<15} {'-'*55:<55} {'-'*11}")

for l in generated_lessons:
    grade    = l["metadata"]["grade_band"]
    ltype    = l["metadata"]["lesson_type"]
    skill    = l["metadata"]["primary_skill"][:52] + "..."
    practice = l["lesson_flow"]["practice"]
    has_scaffold = any(p.get("scaffold") for p in practice)
    print(f"{grade:<8} {ltype:<15} {skill:<55} {'Yes' if has_scaffold else 'No'}")

=== CROSS GRADE BAND COMPARISON ===

Grade    Type            Skill                                                   Scaffolded?
-------- --------------- ------------------------------------------------------- -----------
K-2      Story Retell    Describe a person, place, or thing using sensory det... Yes
6-8      Mission Brief   Present a claim and support it with two pieces of ev... No
9-12     Text Explorer   Analyse how an author builds a sustained argument ac... No


In [ ]:
# Coverage report across all generated grade bands
print("=== COVERAGE REPORTS ===\n")
for grade, domain, _ in grade_requests:
    report = get_coverage_report(grade, domain)
    print(f"  [{grade}] {domain}")
    print(f"    Covered   : {report['covered_count']}/{report['total']}")
    print(f"    Covered   : {report['covered']}")
    print(f"    Remaining : {report['remaining']}")
    print()

=== COVERAGE REPORTS ===

  [K-2] Speaking
    Covered   : 2/5
    Covered   : ['Retell a story in sequence using beginning, middle, and end', 'Describe a person, place, or thing using sensory details']
    Remaining : ['Ask and answer questions about a topic using complete sentences', 'Share an opinion and give one reason to support it', 'Give a two-step oral instruction in the correct order']

  [6-8] Speaking
    Covered   : 1/5
    Covered   : ['Present a claim and support it with two pieces of evidence']
    Remaining : ['Use precise academic vocabulary appropriate to the topic', 'Acknowledge a counterclaim and explain why your position is stronger', 'Organise a spoken argument using a clear claim, evidence, and conclusion', 'Use eye contact and deliberate pacing to engage an audience']

  [9-12] Reading → Speaking
    Covered   : 1/10
    Covered   : ['Analyse how an author builds a sustained argument across an entire text']
    Remaining : ['Construct and deliver a persuasive ar

## Part 6: The Deduplication Registry
Every generation logs its combination to the registry.
Let's see how it's grown across our generations.

In [ ]:
registry = load_registry()
combos   = registry["used_combinations"]

print(f"Registry now contains {len(combos)} entries:\n")
for entry in combos:
    print(f"  [{entry['grade_band']}] {entry.get('ela_domain', 'N/A')} | {entry['theme']}")
    print(f"         Skill     : {entry['skill'][:60]}...")
    print(f"         Lesson ID : {entry['lesson_id']}")
    print(f"         Logged at : {entry.get('generated_at', 'N/A')}")
    print()

Registry now contains 27 entries:

  [3-5] N/A | politics
         Skill     : Explain a process step by step...
         Lesson ID : L-NB-TEST-002
         Logged at : 2026-03-12T17:34:02.094052

  [3-5] N/A | Ocean Ecosystems
         Skill     : Explain a process step by step...
         Lesson ID : L-NB-TEST-002
         Logged at : 2026-03-12T17:34:15.994680

  [3-5] N/A | Ocean Ecosystems
         Skill     : Describe a habitat using sensory details and basic needs...
         Lesson ID : L-35-SPK-009
         Logged at : 2026-03-12T19:54:01.502037

  [9-12] N/A | Climate Change
         Skill     : Analyze an author's use of evidence to support a claim about...
         Lesson ID : L-912-RDG-SPK-009
         Logged at : 2026-03-12T19:55:22.297580

  [9-12] N/A | Persuasive Writing
         Skill     : Analyze and explain the effectiveness of persuasive techniqu...
         Lesson ID : L-912-RDG-SPK-009
         Logged at : 2026-03-12T19:55:53.660668

  [9-12] N/A | Persuasive Wr

## Part 7: Guardrail Demonstration
Let's show the guardrails catching bad inputs
before any API call is made.

In [ ]:
# Bad grade band
print("=== BAD GRADE BAND ===\n")
try:
    gen.generate(
        grade_band = "Year 5",
        ela_domain = "Speaking",
        theme      = "Animals"
    )
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== BAD GRADE BAND ===


[generator] Starting lesson generation
[generator] Grade: Year 5 | Domain: Speaking | Theme: Animals

[generator] Step 1 — Running pre-generation checks...
Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] 'Year 5' is not a valid grade band. Must be one of: ['3-5', '6-8', '9-12', 'K-2']
  ⚠️ [FLAG] Theme 'Animals' is too short. Please provide a descriptive theme.


In [ ]:
# Bad ELA domain
print("=== BAD ELA DOMAIN ===\n")
try:
    gen.generate(
        grade_band = "3-5",
        ela_domain = "Maths",
        theme      = "Animals"
    )
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== BAD ELA DOMAIN ===


[generator] Starting lesson generation
[generator] Grade: 3-5 | Domain: Maths | Theme: Animals

[generator] Step 1 — Running pre-generation checks...
Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] 'Maths' is not a valid ELA domain. Must be one of: ['Listening', 'Reading', 'Reading → Speaking', 'Speaking', 'Writing']
  ⚠️ [FLAG] Theme 'Animals' is too short. Please provide a descriptive theme.


In [ ]:
# Empty theme
print("=== EMPTY THEME ===\n")
try:
    gen.generate(
        grade_band = "3-5",
        ela_domain = "Speaking",
        theme      = ""
    )
except ValueError as e:
    print("Caught ValueError ✅")
    print(e)

=== EMPTY THEME ===


[generator] Starting lesson generation
[generator] Grade: 3-5 | Domain: Speaking | Theme: 

[generator] Step 1 — Running pre-generation checks...
Caught ValueError ✅
[checks] Pre-generation checks failed:
  ⚠️ [FLAG] Theme cannot be empty.


## Part 8: Loading a Previously Generated Lesson
Every lesson is saved to disk. We can reload any lesson
by its ID -- the system has persistent memory across sessions.

In [ ]:
# Load the first lesson we generated in this notebook
first_id = generated_lessons[0]["lesson_id"]
print(f"Loading lesson {first_id} from disk...\n")

reloaded = load_lesson(first_id)

print(f"lesson_id      : {reloaded['lesson_id']}")
print(f"grade_band     : {reloaded['metadata']['grade_band']}")
print(f"primary_skill  : {reloaded['metadata']['primary_skill']}")
print(f"theme          : {reloaded['metadata']['theme']}")
print()
print("Round-trip confirmed ✅ — lesson survived save and reload.")

Loading lesson L-K2-SPK-004 from disk...

lesson_id      : L-K2-SPK-004
grade_band     : K-2
primary_skill  : Describe a person, place, or thing using sensory details
theme          : Nature & Animals

Round-trip confirmed ✅ — lesson survived save and reload.


## Summary: What This Pipeline Demonstrates

| Stage | Module | What it proved |
|---|---|---|
| Pre-checks | `checks.py` | Bad inputs stopped before any API call |
| Skill selection | `skill_selector.py` | Next uncovered taxonomy skill auto-selected |
| Prompt construction | `templates.py` | Skill injected directly — LLM cannot deviate |
| Groq API call | `generator.py` | Free-tier LLM produces structured lessons |
| Validation | `validator.py` | Malformed output caught and corrected |
| Persistence | `file_handler.py` | Lessons saved and reloaded reliably |
| Coverage tracking | `skill_selector.py` | Registry grows, coverage report updates after each generation |

## Research connections visible in the output
- **Narrative frame** (Bruner, 1990) — every lesson has a story hook
- **Worked example** (Sweller, 1994) — model stage shows skill before asking for it
- **Gradual release** (Vygotsky) — practice moves supported → independent
- **Feedback loop** (Hattie & Timperley, 2007) — reflect stage is specific, not generic
- **Grade differentiation** (Sweller, 1988) — K-2 and 9-12 lessons are fundamentally different
- **Skill progression** (Rosenshine, 2012) — one measurable skill per lesson, tracked across generations

## Known limitations
- The `single_skill_check` retains `"and"` detection as a secondary check for edge cases where the LLM rephrases the injected skill
- Cultural bias detection is keyword-based — misses subtle bias
- One few-shot example per grade band — true RAG would use embeddings
- No structured output mode (Groq supports `response_format`) — would eliminate the practice dict bug entirely in production
- Skill cycling is FIFO — a production system would prioritise weak skills based on individual student performance data